# 08 — Single Rigid Body Dynamics

MPCの中心である並進・回転運動方程式を、手計算とCasADi実装で対応づけます。

**前提**: `07_swing_trajectory.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 並進

\[
m\dot v=\sum_{i=1}^{4}c_iF_i+F_{ext}+mg
\]

## 回転

\[
I\dot\omega+\omega\times I\omega
  =R_{B\leftarrow W}\left(\sum_i c_i(r_i-p)\times F_i+\tau_{ext}\right)
\]

SRBDは脚の質量分布や関節運動を直接予測せず、それらを胴体の質量・慣性と
接触力へ縮約します。

In [2]:
# 背景: SRBDの並進式m v_dot=Σc_iF_i+mgでは、接触mask・4脚GRFのshape・重力符号を揃えないと静止条件さえ再現できない。
# 目的: 4脚で重量を均等支持した零加速度と、FLへ20 N加えたときのa_x=20/mを数値検算する。
# 4脚×3軸の床反力配列、総和、近似比較にNumPyを使う。
import numpy as np

# centroidal_model_nominal.py::forward_dynamicsの並進部と同じ変数。
# Go2質量m[kg]と重力加速度の大きさg[m/s^2]を設定する。
mass, g = 15.019, 9.81                 # m [kg], g [m/s^2]
# 4脚すべてを接地c_i=1とするshape (4,)のmaskを作る。
contacts = np.array([1, 1, 1, 1])      # c_i: 接地=1, 遊脚=0
# 各脚が鉛直にmg/4[N]を負担する同一行を複製し、shape (4,3)のGRFを作る。
forces = np.tile([0., 0., mass*g/4], (4, 1))  # F_i [N], 4脚×xyz

# 実装対応: linear_com_acc = (1/mass)*sum(c_i*F_i) + [0,0,-g]
# 数式対応: m*v_dot = sum(c_i F_i) + m*g_vector
# contactsをshape (4,1)へ拡張して各脚の3軸力をmaskし、脚軸方向に総和する。
net_contact_force = (contacts[:, None] * forces).sum(axis=0)
# ΣF/mへworld z負方向の重力[0,0,-g]を加え、COM加速度shape (3,) [m/s^2]を求める。
acc = net_contact_force / mass + np.array([0, 0, -g])
# 四脚の支持力と重力が相殺した加速度を単位付きで表示する。
print("balanced acceleration:", acc, "m/s^2")

# FLへ+20 Nの水平力を加え、a_x=20/mになることを検算する。
# 脚順0=FLのworld x床反力を20 Nへ変更し、水平運動だけを加える。
forces[0, 0] = 20.0
# 更新後のcontact mask付き総床反力shape (3,) [N]を再計算する。
net_contact_force = (contacts[:, None] * forces).sum(axis=0)
# 同じSRBD並進式で外力追加後のCOM加速度[m/s^2]を求める。
acc2 = net_contact_force / mass + np.array([0, 0, -g])
# 20 Nによる約1.332 m/s^2のx加速度を小数3桁で表示する。
print("after +20N at FL:", np.round(acc2, 3), "m/s^2")
# 浮動小数誤差を許容して、重量支持時の3軸加速度が0であることを検査する。
assert np.allclose(acc, 0)
# Newtonの式a_x=F_x/mと数値結果が一致することを検査する。
assert np.isclose(acc2[0], 20.0/mass)

balanced acceleration: [0.00000000e+00 0.00000000e+00 1.77635684e-15] m/s^2
after +20N at FL: [1.332 0.    0.   ] m/s^2


In [3]:
# 背景: 教材上の状態・入力分解が正しくても、CasADi symbolの実次元とずれていればOCPの参照・重み行列とのshape不整合が起きる。
# 目的: 現行nominal centroidal modelを生成し、state=30・input=24・reference=54のCasADi次元を実装から直接確認する。
# 上流のSRBD式とCasADi symbolを構築するnominal modelクラスを読み込む。
from quadruped_pympc.controllers.gradient.nominal.centroidal_model_nominal import Centroidal_Model_Nominal
# デフォルト設定で現行モデルを生成し、states・inputs・y_ref symbolへアクセス可能にする。
model = Centroidal_Model_Nominal()
# 状態symbolの第1次元を表示し、COM・姿勢・足位置・積分状態の合計30を確認する。
print("CasADi state dim:", model.states.size1())
# 入力symbolの第1次元を表示し、足速度12+GRF12=24を確認する。
print("CasADi input dim:", model.inputs.size1())
# stage referenceがstate30+input24=54要素であることを表示する。
print("reference dim   :", model.y_ref.size1())
# 現行state symbolが教材のshape契約30と一致しない変更を即座に検出する。
assert model.states.size1() == 30
# 現行input symbolが4脚×(足速度3+GRF3)=24と一致することを検査する。
assert model.inputs.size1() == 24

CasADi state dim: 30
CasADi input dim: 24
reference dim   : 54


実装では接触フラグが力とモーメントの両方をmaskします。
遊脚に大きなGRF変数が入っても運動へ寄与しませんが、後段Interfaceでも再度maskされます。
二重防御の場所を区別してください。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。